# Task A：编号数组指针斜率验证

**已完成：** 寄存器探针 0/100、1/100；64 条诊断未发现输入错配或提交批量导致的答案差异。
最新指针探针 delta=11/100、lookup=87/100（提交 `d2b57a4`）；原文已归档，均无解析失败/截断。
本轮不重复探针、不调整题目或采样，直接检验 lookup 的 CoT 深度曲线。

**运行本轮：**
1. Colab/T4 保持默认 `RUN_STAGE="slope"`、`SELECTED="pointer40_lookup"`，从头运行。
2. 已归档的真实探针自动恢复并验证；本轮新增 d=1,5,10,15,20 ×20题×DA/CoT，共 200 条。
3. 查看 CoT(20)、DA(20)、gap 变化与趋势；下载最后一格 ZIP，保存 notebook 到 GitHub。
4. 实测 CoT(20)∈[0.40,0.75] 且经人工审查后才允许正式生成；当前 `GENERATE_FINAL=False`。

lookup 的 DA(1)=0.87 在任务表描述目标内，但未命中启发式探针窗口。
继续小样本的理由已明确记录，不将预测当实测，也不自动批准正式生成。
`search` 保留上一轮有界策略供复现；`offline` 只验证代码。


## Colab / T4 准备

在 Colab 选择 **T4 GPU**，可直接从 GitHub 打开本文件；无需先克隆仓库。
配置 cell 优先使用 `TASK_A_ROOT` 或已存在的仓库，否则自动创建 `/content/taskA` 作为实验目录。
不要在 CPU 环境执行加载 cell。
仓库顶层的依赖面向 Python 3.14，本实验只需以下 notebook 专用依赖，沿用 Colab 自带 Python。
下方依赖准备 cell 会在真实推理模式下安装缺失包；也可提前手动安装：

```python
%pip install vllm datasets nbformat
```

可选：如果需要完整仓库，可先克隆再运行配置 cell：

```python
!git clone --branch taskA https://github.com/wangziwei2018/CISC7404-Project.git /content/CISC7404-Project
%cd /content/CISC7404-Project
```

上传本 notebook 也支持相同的独立运行方式。需要自定义保存位置时，在配置 cell **之前**执行：

```python
import os
os.environ["TASK_A_ROOT"] = "/content/taskA"
```

指定目录不存在时会自动创建。`/content` 是临时存储，运行时删除后文件会丢失，请及时下载结果。
模型使用 `Qwen/Qwen3-4B-Instruct-2507`，需要访问 Hugging Face 下载权重；不改成 base 或 Thinking。
推荐把 `TASK_A_ROOT` 设为 Google Drive 中的项目目录以保存断点；本 notebook 不自动挂载 Drive。
若依赖安装 cell 提示重启，只需重启会话后重新从头运行（不要“断开连接并删除运行时”）。
**仅“将 notebook 保存到 GitHub”不会上传运行时内的 JSONL。** 跑完后下载最后一格导出的
`task_a_results.zip`，解压后把 `Experiments/task_a/` 下的结果文件提交到 `taskA`，再保存 notebook。
真实结果：`Experiments/task_a/calib_v3_task.jsonl`；正式数据：
最终选中指针任务时为 `Experiments/data_generation/synthesis_pointer_numeric_dataset.jsonl`；
中间题目位于 `work/task_a/`，search/诊断/斜率摘要位于 `Experiments/task_a/`。

In [1]:
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import re
import tempfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

def project_root():
    if os.environ.get("TASK_A_ROOT"):
        root = Path(os.environ["TASK_A_ROOT"]).expanduser().resolve()
        root.mkdir(parents=True, exist_ok=True)
        return root
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "task.md").is_file():
            return parent
    content = Path("/content")
    colab = content / "CISC7404-Project"
    if (colab / "task.md").is_file():
        return colab
    if content.is_dir():
        # 从 GitHub 直接打开 Colab 只载入 notebook，不会克隆仓库。
        # 本实验自包含；使用独立输出目录，无需下载其它仓库文件。
        root = content / "taskA"
        root.mkdir(parents=True, exist_ok=True)
        print(f"未发现仓库，使用 Colab 实验目录：{root}。运行时删除后此目录会丢失，请及时保存结果。")
        return root
    raise RuntimeError("请切到仓库目录，或先运行 os.environ['TASK_A_ROOT'] = '可写实验目录'。")

ROOT = project_root()
WORK_DIR = ROOT / "work" / "task_a"
RESULT_PATH = ROOT / "Experiments" / "task_a" / "calib_v3_task.jsonl"
DATA_PATH = ROOT / "Experiments" / "data_generation" / "synthesis_register_dataset.jsonl"
VERSION = "register_v3.1"
SEED = 7404
D_LIST = (1, 5, 10, 15, 20)
RUN_STAGE = "slope"  # @param ["search", "diagnostic", "probe", "slope", "offline"]
if RUN_STAGE not in ("search", "diagnostic", "probe", "slope", "offline"):
    raise ValueError("RUN_STAGE 只允许 search / diagnostic / probe / slope / offline")
RUN_SEARCH = RUN_STAGE == "search"
RUN_DIAGNOSTIC = RUN_STAGE == "diagnostic"
RUN_PROBES = RUN_STAGE == "probe"
RUN_SLOPE = RUN_STAGE == "slope"
AUTO_DOWNLOAD = True  # Colab 运行结束时自动下载结果包；本地只显示路径
CURRENT_JOBS = []
PROBE_REPORTS = {}
PROBE_REPORT_PATH = RESULT_PATH.parent / "task_a_probe_summary.json"
SLOPE_REPORT_PATH = RESULT_PATH.parent / "task_a_summary.json"
DIAGNOSTIC_PATH = RESULT_PATH.parent / "task_a_diagnostic.jsonl"
DIAGNOSTIC_REPORT_PATH = RESULT_PATH.parent / "task_a_diagnostic_summary.json"
print(f"本次运行阶段：{RUN_STAGE}；" + ("只验证代码，不产生真实校准结果。" if RUN_STAGE == "offline" else "将执行真实模型推理。"))
GENERATE_FINAL = False
PROBE_CANDIDATES = ["pointer40_delta"]  # 手动 probe 模式；search 按预先声明的规则选择后续配置
SELECTED = "pointer40_lookup"
PROBE_OVERRIDE_REASON = "lookup 的真实 DA(1)=87/100，位于 task.md A.0 的 0.80–0.95 描述目标内；0.955–0.986 仅由把 DA 单步率当 CoT 独立逐步率的启发式推导而来，不构成 CoT(20) 实测。先做一次独立 200 条斜率直接检验唯一硬门槛；本例外不批准正式生成。"
RESTORE_ARCHIVED_PROBE = True  # slope 开始前恢复已归档的真实探针，SHA256 校验，保留本地结果
REVIEW = None  # 看完斜率结果和原文后，在末尾审核 cell 填写
N_FINAL = 100  # 每个串行深度、每个并行宽度的题数；正式题与校准题完全分离
SEARCH_START = "pointer40_delta"
SEARCH_MAX_PROBES = 3  # 最多 300 条探针；命中窗口后至多再跑 200 条斜率验证
SEARCH_REPORT_PATH = RESULT_PATH.parent / "task_a_search_summary.json"

TIER = "B"  # B：4B / T4 正式校准；A：1.7B / RTX3070 仅链路自测
MODEL_REVISION = None  # 留空即可：加载前自动解析 HF commit；也可填 tag/分支或完整 commit
PARAMS = {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "max_tokens": 6144}
# top_k 只记录、不传入 SamplingParams，遵守 AGENTS.md。
BATCH_SIZE = 16

def digest(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

def stable_seed(*parts):
    return int(digest(parts)[:8], 16)

def read_jsonl(path):
    if not Path(path).exists():
        return []
    rows = []
    with Path(path).open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"{path}:{line_no} 有不完整 JSON；备份并修复该行后恢复，不静默丢弃。") from exc
    return rows

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # 完整写入后替换，避免生成到一半留下貌似完整的题集。
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    tmp.replace(path)

def roundtrip_questions(path, questions):
    from datasets import load_dataset
    write_jsonl(path, questions)
    # datasets 回读是实际推理的数据入口，不直接用内存中的原始题目。
    ds = load_dataset("json", data_files=str(path), split="train", cache_dir=str(WORK_DIR / "hf_cache"))
    return ds.to_list()

未发现仓库，使用 Colab 实验目录：/content/taskA。运行时删除后此目录会丢失，请及时保存结果。
本次运行阶段：slope；将执行真实模型推理。


## 依赖准备与 GPU 检查

真实诊断/探针/斜率模式下先确认有 GPU，再安装缺失的 vLLM / datasets / nbformat。
**安装成功不一定需要重启**：仅当安装实际改变了已加载包的版本，或内存中的包版本与磁盘不一致时，
才显示 `RESTART_REQUIRED` 并停止。提示中会列出具体版本变化。
此时在 Colab 菜单选择 **代码执行程序 → 重新启动会话**，然后再次“全部运行”；
不要选择“断开连接并删除运行时”，否则已安装依赖和输出文件也会被删除。
重启前反复运行本格不会绕过保护；重启后会自动继续。

`offline` 模式只需 datasets，既不安装 vLLM，也不加载模型。
本格通过只是环境就绪，当前轮次默认 slope：恢复已完成的 lookup 探针后，新增 200 条斜率；不运行正式全矩阵。

In [2]:
def dependency_versions(names):
    versions = {}
    for name in sorted(names):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def loaded_dependency_state():
    import sys
    from packaging.version import InvalidVersion, Version

    package_map = importlib.metadata.packages_distributions()
    roots = {name.split(".")[0] for name in sys.modules}
    names = {dist for root in roots for dist in package_map.get(root, [])}
    installed = dependency_versions(names)
    stale = {}
    # 识别上一轮安装后未重启的会话；只比较可明确对应单一发行包的版本。
    for root in sorted(roots):
        distributions = package_map.get(root, [])
        if len(distributions) != 1:
            continue
        name = distributions[0]
        module = sys.modules.get(root)
        memory = vars(module).get("__version__") if module is not None else None
        disk = installed[name]
        if not isinstance(memory, str) or disk is None:
            continue
        try:
            # PyTorch 的 +cuXXX 后缀可能只出现在模块版本中，不因此误判需重启。
            different = Version(memory).public != Version(disk).public
        except InvalidVersion:
            continue
        if different:
            stale[name] = {"loaded": memory, "installed": disk}
    return installed, stale


def dependency_kernel_token():
    import sys
    import uuid

    # sys 在同一内核内稳定，重跑配置 cell 不会重置；重启后产生新 token。
    if not hasattr(sys, "_task_a_dependency_kernel_token"):
        sys._task_a_dependency_kernel_token = uuid.uuid4().hex
    return sys._task_a_dependency_kernel_token


def prepare_dependencies(required, work_dir=WORK_DIR):
    import subprocess
    import sys

    marker = Path(work_dir) / "dependency_restart_required.json"
    kernel_token = dependency_kernel_token()
    restart_message = (
        "RESTART_REQUIRED：请在 Colab 选择‘代码执行程序 → 重新启动会话’，然后从头运行。"
        "不要删除运行时，也不要只重复执行当前 cell。"
    )
    if marker.is_file():
        previous = json.loads(marker.read_text(encoding="utf-8"))
        if previous["kernel_token"] == kernel_token:
            print("本会话尚未重启：", json.dumps(previous["changes"], ensure_ascii=False))
            raise RuntimeError(restart_message)
        marker.unlink()
        print("已检测到新会话，继续检查依赖。")

    def require_restart(changes):
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps({"kernel_token": kernel_token, "changes": changes},
                                     ensure_ascii=False, indent=2), encoding="utf-8")
        print("需重新加载的包：", json.dumps(changes, ensure_ascii=False, indent=2))
        raise RuntimeError(restart_message)

    before, stale = loaded_dependency_state()
    if stale:
        require_restart(stale)
    missing = [name for name in required if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        importlib.invalidate_caches()
        after = dependency_versions(before)
        changed = {name: {"before": version, "after": after[name]}
                   for name, version in before.items() if after[name] != version}
        print("依赖安装成功。")
        if changed:
            require_restart(changed)
        print("已加载包的版本未改变，无需重启。")
    print("依赖版本：", json.dumps(dependency_versions(required), ensure_ascii=False))


if RUN_SEARCH or RUN_DIAGNOSTIC or RUN_PROBES or RUN_SLOPE:
    import shutil
    import subprocess

    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("未检测到 GPU 驱动。请在 Colab 选择 T4 GPU；仅检查代码时显式选 RUN_STAGE='offline'。")
    gpu_check = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    if not gpu_check.stdout.strip():
        raise RuntimeError("未检测到 GPU。请选择 Colab T4 GPU 后重试。")
    print(gpu_check.stdout.strip())
    prepare_dependencies(("vllm", "datasets", "nbformat"))
    print("GPU 与依赖检查完成；继续运行模型加载和真实探针。")
else:
    print("offline 模式：跳过 GPU 依赖安装。")

Tesla T4, 15360 MiB
依赖版本： {"datasets": "4.8.5", "nbformat": "5.11.1", "vllm": "0.33.0"}
GPU 与依赖检查完成；继续运行模型加载和真实探针。


## 1. 历史候选：寄存器任务与工作量（保留复核）

`record.md` 的示例混合了单更新和条件更新，与“每步同构”的文字不一致。本版固定每一步：
读取更新前的两个指定寄存器；比较大小；按分支对一个寄存器减 x、另一个加 y，或反向更新。
**两次更新同时生效，等号走 otherwise；x、y 独立且不相等**，避免 `A+B` 守恒导致答案可抄近路。
所有状态在 `[0, base−1]`，不做 mod、截断、min/max 或结果聚合。

原首选为两位、双寄存器，答案为 `A*100+B`。三寄存器备选追踪额外状态，编码为
`A*10000+B*100+C`；改变位数时 `base=10**digits`，不会使用有歧义的固定乘数。
比较的寄存器对可以不同，但每步都是同一算子。用更多状态/位数调难度，不混入更难算子。
生成器用全值域初态（排除无法合法更新的全零/全上界）和有界拒绝采样，边界会影响操作分布；
必须在校准中检查，不能从 p1 推断成功。

`W` 沿用任务提案的**复合寄存器更新次数**；一次含比较和两次加减，另记录
`W_atomic=3W`。最终答案的无歧义编码是读出协议，不计作更新，串行/并行一致。

并行版采用 `w` 条独立链，长度 `[1,2,...,2]`：`W=2w−1`，真实深度为 `1`（w=1）或 `2`。
与串行 `d=2w−1` 的更新工作量严格相同；**没有为了套公式再添加聚合步骤**。
串行覆盖 d=1..20，并行匹配其中奇数 d=1..19（w=1..10），w=1 是退化对照。
并行答案依次读出所有链的编码，避免漏掉某条链的工作。多答案会增加读出成本，
需同时看 `per_branch_ok`，并在人工审核中评估这一混杂，不能声称只改变了深度。

In [3]:
@dataclass(frozen=True)
class TaskConfig:
    name: str
    digits: int = 2
    registers: int = 2
    delta_max: int = 9

    def __post_init__(self):
        if self.digits not in (1, 2, 3) or self.registers not in (2, 3):
            raise ValueError("只支持 1..3 位、2/3 个寄存器")
        if not 2 <= self.delta_max < 10 ** self.digits:
            raise ValueError("delta_max 必须至少为 2 且小于编码基数")

    @property
    def base(self):
        return 10 ** self.digits

CANDIDATES = {
    c.name: c for c in (
        TaskConfig("register2_small"),
        TaskConfig("register2_wide", delta_max=19),
        TaskConfig("register3_small", registers=3),
        TaskConfig("register2_easy", digits=1, delta_max=3),
    )
}

def encode_state(state, base):
    if any(not 0 <= value < base for value in state):
        raise ValueError("状态越界，不能无歧义编码")
    result = 0
    for value in state:
        result = result * base + value
    return result

def apply_op(state, op):
    out = list(state)
    left, right, x, y = (op[k] for k in ("left", "right", "x", "y"))
    if state[left] > state[right]:
        out[left] -= x
        out[right] += y
    else:
        out[left] += x
        out[right] -= y
    return out

def make_chain(rng, steps, config):
    # 全零 / 全上界没有合法的一加一减；仅排除这两种不可行初态。
    while True:
        state = [rng.randrange(config.base) for _ in range(config.registers)]
        if len(set(state)) > 1 or state[0] not in (0, config.base - 1):
            break
    initial, ops, trace = list(state), [], []
    for _ in range(steps):
        for attempt in range(10000):
            left, right = rng.sample(range(config.registers), 2)
            x, y = rng.sample(range(1, config.delta_max + 1), 2)
            op = {"left": left, "right": right, "x": x, "y": y}
            updated = apply_op(state, op)
            if all(0 <= v < config.base for v in updated):
                break
        else:
            raise RuntimeError("有界生成失败，请检查配置；禁止静默截断或取模")
        ops.append(op)
        trace.append(list(updated))
        state = updated
    return {"initial": initial, "ops": ops, "trace": trace,
            "final": state, "ans": encode_state(state, config.base)}

def op_text(op):
    left, right = ("ABC"[op[k]] for k in ("left", "right"))
    x, y = op["x"], op["y"]
    return (f"If {left} > {right}, subtract {x} from {left} and add {y} to {right}; "
            f"otherwise add {x} to {left} and subtract {y} from {right}.")

def render_question(chains, config):
    names = "ABC"[:config.registers]
    formula = " + ".join(
        name if power == 0 else f"{name}*{config.base ** power}"
        for name, power in zip(names, reversed(range(config.registers)))
    )
    lines = [
        "Process the operations in the listed order within each independent chain.",
        "For each operation, compare the two values BEFORE either update; apply both updates simultaneously.",
        "Equality uses the otherwise branch. Registers not named in an operation stay unchanged.",
        "Use ordinary integer arithmetic, with no wrapping or clipping.",
    ]
    for index, chain in enumerate(chains, 1):
        lines.append(f"Chain {index}: " + ", ".join(f"{n}={v}" for n, v in zip(names, chain["initial"])))
        lines.extend(f"{i}. {op_text(op)}" for i, op in enumerate(chain["ops"], 1))
    lines.append(f"Encode the final registers of each chain as {formula}.")
    lines.append("What is the final encoded integer?" if len(chains) == 1 else
                 "What are the final encoded integers, in chain order? Return all of them, separated by commas.")
    return "\n".join(lines)

def lookup_fields(chains, base):
    # 查表字段只来自题干：不暴露最终状态，不暗中添加 k 轴约束。
    return {
        "START": [encode_state(c["initial"], base) for c in chains],
        "OPS": [len(c["ops"]) for c in chains],
        "FIRST": [op_text(c["ops"][0]) for c in chains],
        "LAST": [op_text(c["ops"][-1]) for c in chains],
        "MAXDELTA": [max(max(op["x"], op["y"]) for op in c["ops"]) for c in chains],
        # 不沿用 min-chain 的 NEG：新任务没有负数初态/字面负增量，定义将空转。
    }

def make_register_question(config, d, i, split, kind="serial", w=1):
    if not 1 <= d <= 20 or i < 0 or not split:
        raise ValueError("无效的深度、题号或 split")
    if kind == "serial":
        if w != 1:
            raise ValueError("串行宽度必须为 1")
        lengths = [d]
    elif kind == "parallel":
        if not 1 <= w <= 10 or d != (1 if w == 1 else 2):
            raise ValueError("并行深度必须等于最长链长度，w∈1..10")
        lengths = [1] + [2] * (w - 1)
    else:
        raise ValueError("未知任务布局")
    key = {"version": VERSION, "seed": SEED, "config": asdict(config), "split": split,
           "kind": kind, "d": d, "w": w, "i": i}
    rng = random.Random(stable_seed(key))
    chains = [make_chain(rng, steps, config) for steps in lengths]
    workload = sum(lengths)
    answers = [chain["ans"] for chain in chains]
    return {**key, "qid": digest(key), "W": workload, "W_atomic": 3 * workload,
            "matched_serial_d": workload, "chain_lengths": lengths,
            "ans": ",".join(map(str, answers)), "answers": answers,
            "q": render_question(chains, config), "chains": chains,
            "lookup": lookup_fields(chains, config.base)}

## 当前候选：编号数组指针

诊断的基础及独立编码案例全部正确，64 条输入均对齐，32 对单条/批量请求的最终答案一致。
同一批寄存器组合题 DA 0/4、CoT 4/4，直接输出 A/B 时 DA 1/2。两种提交批量是重复对照，不能当作双倍独立样本。
现有证据支持放弃当前寄存器配置，按 task.md 的第二优先级转向编号数组指针；不继续扩大旧任务的深度矩阵。

每题固定 40 个编号。首选规则为 `current = current + delta[current]`，每步查一次表、做一次加法；
答案就是最终编号，**没有寄存器组合、答案编码或 aggregation**。所有访问均在表内，不做 mod。
难度旋钮是编号位数和表行顺序；若 delta 单步仍过难，使用每步只查表的 `current = next[current]` 变体。
每种配置内部的步骤同构，不在一道题里混用算子。

串行路径不重复，覆盖 d=1..20；并行共用同一张只读表，w 条路径的节点完全不重叠，
长度为 `[1,2,...,2]`，因此实际 d≤2、W=2w−1，与串行奇数深度配平。w≤10 时最多用 29 个不同节点，40 行足够。
W 表示转移次数；delta 的 `W_atomic=2W`（查表+加法），lookup 的 `W_atomic=W`。不可跨变体混用 W_atomic。
最终读出所有并行链的编号，仍需在人工审查中评估多答案读出成本。

START / OPS / TABLE_SIZE / INDEX_MIN / INDEX_MAX / FIRST / LAST 来自输入；FIRST/LAST 明确指印出的首/末表行。
这些字段只是给 Task B 留接口，不表示旧约束可直接迁移。正式数据须在校准与人工审核后生成。


In [4]:
POINTER_VERSION = "pointer_numeric_v1"

@dataclass(frozen=True)
class PointerConfig:
    name: str
    table_size: int = 40
    index_offset: int = 0
    operation: str = "delta"
    shuffle_rows: bool = False
    family: str = "pointer_numeric"

    def __post_init__(self):
        if self.table_size != 40 or self.index_offset < 0 or self.operation not in ("delta", "lookup"):
            raise ValueError("当前指针校准固定 40 个编号，offset≥0，operation 为 delta/lookup")
        if self.family != "pointer_numeric":
            raise ValueError("任务族标记不一致")

POINTER_CANDIDATES = {
    config.name: config for config in (
        PointerConfig("pointer40_delta"),
        PointerConfig("pointer40_delta3", index_offset=100),
        PointerConfig("pointer40_delta2", index_offset=10),
        PointerConfig("pointer40_delta3_shuffled", index_offset=100, shuffle_rows=True),
        PointerConfig("pointer40_lookup", operation="lookup"),
        PointerConfig("pointer40_lookup_shuffled", operation="lookup", shuffle_rows=True),
    )
}
CANDIDATES.update(POINTER_CANDIDATES)


def config_from_dict(value):
    return PointerConfig(**value) if value.get("family") == "pointer_numeric" else TaskConfig(**value)


def render_pointer_question(table, chains, config):
    end = config.index_offset + config.table_size - 1
    value_name = "delta" if config.operation == "delta" else "next"
    rule = ("At each transition, read delta at the row whose index equals current, then set current = current + delta."
            if config.operation == "delta" else
            "At each transition, read next at the row whose index equals current, then replace current with that next value.")
    lines = [f"The fixed table has {config.table_size} rows, with integer indices {config.index_offset} through {end}.",
             rule, "Repeat the transition exactly the specified number of times. The table never changes.",
             "Use ordinary integers without wrapping or clipping. All required indices exist in the table.",
             f"index | {value_name}"]
    lines.extend(f"{row['index']} | {row['value']}" for row in table)
    for j, chain in enumerate(chains, 1):
        lines.append(f"Chain {j}: initial current = {chain['initial']}; number of transitions = {chain['steps']}.")
    lines.append("What is the final current index?" if len(chains) == 1 else
                 "The chains are independent. Return all final current indices in chain order, separated by commas.")
    return "\n".join(lines)


def make_pointer_question(config, d, i, split, kind="serial", w=1):
    if not 1 <= d <= 20 or i < 0 or not split:
        raise ValueError("无效的深度、题号或 split")
    if kind == "serial" and w == 1:
        lengths = [d]
    elif kind == "parallel" and 1 <= w <= 10 and d == (1 if w == 1 else 2):
        lengths = [1] + [2] * (w - 1)
    else:
        raise ValueError("无效的串行/并行布局")
    key = {"version": POINTER_VERSION, "seed": SEED, "config": asdict(config), "split": split,
           "kind": kind, "d": d, "w": w, "i": i}
    rng = random.Random(stable_seed(key))
    indices = list(range(config.index_offset, config.index_offset + config.table_size))
    # 含起点、终点的所有路径节点不重叠，避免循环捷径和并行重复工作。
    reserved = rng.sample(indices, sum(length+1 for length in lengths))
    edges, chains, offset = {}, [], 0
    for length in lengths:
        path = reserved[offset:offset+length+1]
        offset += length+1
        edges.update(zip(path, path[1:]))
        chains.append({"initial": path[0], "steps": length, "trace": path[1:], "final": path[-1], "ans": path[-1]})
    for index in indices:
        if index not in edges:
            edges[index] = rng.choice([other for other in indices if other != index])
    table = [{"index": index, "value": edges[index]-index if config.operation == "delta" else edges[index]}
             for index in indices]
    if config.shuffle_rows:
        rng.shuffle(table)
    workload = sum(lengths)
    answers = [chain["final"] for chain in chains]
    lookup = {"START": [chain["initial"] for chain in chains], "OPS": lengths,
              "TABLE_SIZE": config.table_size, "INDEX_MIN": indices[0], "INDEX_MAX": indices[-1],
              "FIRST": f"{table[0]['index']} | {table[0]['value']}",
              "LAST": f"{table[-1]['index']} | {table[-1]['value']}"}
    return {**key, "task_family": "pointer_numeric", "qid": digest(key), "W": workload,
            "W_atomic": workload * (2 if config.operation == "delta" else 1),
            "matched_serial_d": workload, "chain_lengths": lengths, "ans": ",".join(map(str, answers)),
            "answers": answers, "q": render_pointer_question(table, chains, config),
            "table": table, "chains": chains, "lookup": lookup}


def make_question(config, d, i, split, kind="serial", w=1):
    generator = make_pointer_question if isinstance(config, PointerConfig) else make_register_question
    return generator(config, d, i, split, kind, w)


def verify_pointer_question(question):
    config = PointerConfig(**question["config"])
    table = {row["index"]: row["value"] for row in question["table"]}
    allowed = set(range(config.index_offset, config.index_offset + config.table_size))
    assert len(question["table"]) == config.table_size and set(table) == allowed
    for index, value in table.items():
        target = index + value if config.operation == "delta" else value
        assert target in allowed and target != index
    visited_all, answers, lengths = set(), [], []
    # 独立按表执行，不读取生成时的 edges，也不借用保存的 trace 求答案。
    for chain in question["chains"]:
        current = chain["initial"]
        visited, trace = [current], []
        for _ in range(chain["steps"]):
            current = current + table[current] if config.operation == "delta" else table[current]
            visited.append(current)
            trace.append(current)
        assert len(set(visited)) == len(visited)
        assert not visited_all.intersection(visited)
        visited_all.update(visited)
        assert trace == chain["trace"] and current == chain["final"] == chain["ans"]
        answers.append(current)
        lengths.append(chain["steps"])
    assert answers == question["answers"] and question["ans"] == ",".join(map(str, answers))
    assert question["d"] == max(lengths) and question["w"] == len(lengths)
    assert question["W"] == sum(lengths) and question["chain_lengths"] == lengths
    assert question["W_atomic"] == question["W"] * (2 if config.operation == "delta" else 1)
    assert question["matched_serial_d"] == question["W"]
    if question["kind"] == "parallel":
        assert question["W"] == 2*question["w"]-1
    assert question["lookup"]["START"] == [chain["initial"] for chain in question["chains"]]
    assert question["lookup"]["OPS"] == lengths
    assert question["lookup"]["TABLE_SIZE"] == len(table)
    assert question["lookup"]["INDEX_MIN"] == min(table) and question["lookup"]["INDEX_MAX"] == max(table)
    first, last = question["table"][0], question["table"][-1]
    assert question["lookup"]["FIRST"] == f"{first['index']} | {first['value']}"
    assert question["lookup"]["LAST"] == f"{last['index']} | {last['value']}"
    assert question["q"] == render_pointer_question(question["table"], question["chains"], config)


def offline_pointer_tests():
    count = 0
    for config in POINTER_CANDIDATES.values():
        for d in range(1, 21):
            for i in range(5):
                question = make_question(config, d, i, "selftest")
                verify_pointer_question(question)
                assert question == make_question(config, d, i, "selftest")
                count += 1
        for w in range(1, 11):
            for i in range(5):
                verify_pointer_question(make_question(config, 1 if w == 1 else 2, i, "selftest", "parallel", w))
                count += 1
    print(f"指针离线自测通过：{count} 道题独立按表复算；无模型结果、无正式数据输出。")


offline_pointer_tests()

指针离线自测通过：900 道题独立按表复算；无模型结果、无正式数据输出。


## 2. Prompt、答案判分与作业指纹

DA/CoT 只由 system prompt 控制；共同使用 `####ANSWER=` 标记，未加入 k 轴约束。
数值正确性与格式分开：允许最终裸整数/整数列表，空格或 `.0` 不改变数值；
保留多个答案标记、DA 多写推理等诊断，**不据此筛除答案正确的样本**。
没有可辨识最终答案时计错并记录 `parse_ok=False`，不在中间步骤里搜索碰巧等于 gold 的数。

In [5]:
PREAMBLE = {
    "DA": "Solve silently. Do not show calculations, intermediate values, explanations or comments. Output only the final answer line.\n",
    "CoT": "Solve step by step. Explicitly write the register values after every operation, one operation per line, identifying the chain. Then give the final answer.\n",
}
BASE_RULE = "The final line must start with ####ANSWER= followed by the requested encoded integer(s), with nothing after them."

def build_messages(question, mode):
    if mode not in PREAMBLE:
        raise ValueError("只允许 DA / CoT")
    if question.get("task_family") == "pointer_numeric":
        preamble = (PREAMBLE["DA"] if mode == "DA" else
                    "Solve step by step. Explicitly write the current index after every transition, one transition per line, identifying the chain. Then give the final answer.\n")
        rule = "The final line must start with ####ANSWER= followed by the final index (or indices in chain order, separated by commas), with nothing after them."
        return [{"role": "system", "content": preamble + rule}, {"role": "user", "content": question["q"]}]
    rule = ("The final line must start with ####ANSWER= followed by the requested integer(s), with nothing after them."
            if question.get("diagnostic_kind") and question["diagnostic_kind"] != "register_encoded" else BASE_RULE)
    return [{"role": "system", "content": PREAMBLE[mode] + rule},
            {"role": "user", "content": question["q"]}]

def parse_answer(text, width):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    marked = re.findall(r"(?m)^\s*####ANSWER\s*=\s*([^\n]*)", text)
    payload = marked[-1].strip() if marked else (lines[-1] if lines else "")
    number = r"[+-]?\d+(?:\.0+)?"
    if not re.fullmatch(number + rf"(?:\s*,\s*{number})*", payload):
        return None
    values = [int(part.strip().split(".")[0]) for part in payload.split(",")]
    return values if len(values) == width else None

def score_answer(text, question, mode):
    values = parse_answer(text, question.get("answer_width", question["w"]))
    expected = question["answers"]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    marked = re.findall(r"(?m)^\s*####ANSWER\s*=", text)
    return {
        "answer_ok": values == expected,
        "per_branch_ok": [values is not None and a == b for a, b in zip(values or [None] * len(expected), expected)],
        "parse_ok": values is not None, "predicted": values,
        "base_ok": bool(lines) and lines[-1].startswith("####ANSWER=") and len(marked) == 1 and values is not None,
        "multiple_answers": len(marked) > 1,
        "da_extra_text": mode == "DA" and len(lines) > 1,
    }

def make_jobs(questions, stage, modes, run_meta, execution=None):
    if stage not in ("diagnostic", "probe", "slope"):
        raise ValueError("只支持诊断、探针和斜率阶段")
    jobs = []
    for question in questions:
        for mode in modes:
            messages = build_messages(question, mode)
            seed = stable_seed(question.get("version", VERSION), SEED, question["qid"], mode)
            spec = {"stage": stage, "question": question, "mode": mode, "messages": messages,
                    "sampling": PARAMS, "sampling_seed": seed, "run_meta": run_meta}
            if execution is not None:
                spec["execution"] = execution
            jobs.append({**spec, "id": digest(spec)})
    if len({j["id"] for j in jobs}) != len(jobs):
        raise ValueError("作业中有重复题目")
    return jobs

def index_results(path):
    indexed = {}
    for row in read_jsonl(path):
        if row["id"] in indexed:
            raise ValueError(f"结果文件存在重复 id：{row['id']}")
        indexed[row["id"]] = row
    return indexed

def load_complete(jobs, path=RESULT_PATH):
    indexed = index_results(path)
    missing = [job["id"] for job in jobs if job["id"] not in indexed]
    if missing:
        raise RuntimeError(f"当前配置缺少 {len(missing)}/{len(jobs)} 条真实结果，请先跑完对应阶段。")
    rows = [indexed[job["id"]] for job in jobs]
    for job, row in zip(jobs, rows):
        if any(row.get(key) != value for key, value in job.items()):
            raise ValueError("结果与作业指纹不一致")
        if row.get("backend") != "vllm" or job["run_meta"].get("tier") != "B":
            raise ValueError("正式筛选只接受档位 B 的真实 vLLM 结果，离线测试/档位 A 不可定题")
        if job["stage"] != "diagnostic" and row.get("input_tokens_match") is False:
            raise ValueError("推理收到的 token 与预期题干不一致；该结果不能用于校准，请先诊断。")
        if any(row.get(key) != value for key, value in score_answer(row["text"], job["question"], job["mode"]).items()):
            raise ValueError("判分记录与原文不一致")
    return rows

## 3. 统计与阶段门槛（不筛掉截断或格式错误）

`task.md` 同时给出 DA(1)=0.80–0.95 与探针窗口 0.955–0.986，两者不相交。
本版分别展示，**不把两者合成不可能通过的硬门槛**。p1 只用作筛选启发式：
DA 单步正确率包含编码/读出错误，不能当作 CoT 的独立同分布逐步成功率。
100 题下窗口只对应 96/97/98 题正确，附 Wilson 95% CI，`p1**20` 仅标为预测。

斜率阶段使用与探针不同的题，每个深度 DA/CoT 配对使用同一道题。汇报
CoT(20)、DA(20)、gap(20)−gap(1)、各深度准确率、解析/截断率和原文。
Cochran–Armitage 使用实际深度分值，对 **CoT 准确率随深度下降** 作双侧检验，
要求 z<0 且 p<0.01；它不是 gap 趋势检验，更不能单凭显著性证明严格单调。
额外报告 gap 的相邻变化及配对 bootstrap 端点差区间，小样本结果须人工判断。

In [6]:
def wilson(successes, n, z=1.959963984540054):
    if n <= 0 or not 0 <= successes <= n:
        raise ValueError("无效二项计数")
    p = successes / n
    denom = 1 + z*z / n
    centre = (p + z*z / (2*n)) / denom
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return [max(0.0, centre-radius), min(1.0, centre+radius)]

def ca_trend(successes, totals, scores):
    if not (len(successes) == len(totals) == len(scores)) or not totals:
        raise ValueError("趋势数组长度不一致")
    if any(n <= 0 or not 0 <= x <= n for x, n in zip(successes, totals)):
        raise ValueError("无效趋势计数")
    n = sum(totals)
    p = sum(successes) / n
    mean = sum(t*s for t, s in zip(totals, scores)) / n
    variance = p*(1-p)*sum(t*(s-mean)**2 for t, s in zip(totals, scores))
    if variance == 0:
        return {"z": 0.0, "p_two_sided": 1.0, "testable": False}
    z = sum((s-mean)*(x-t*p) for x, t, s in zip(successes, totals, scores)) / math.sqrt(variance)
    return {"z": z, "p_two_sided": math.erfc(abs(z)/math.sqrt(2)), "testable": True}

def probe_summary(rows):
    if len(rows) != 100 or any(r["stage"] != "probe" or r["mode"] != "DA" or r["question"]["d"] != 1 for r in rows):
        raise ValueError("探针必须为 d=1 × 100 × DA")
    hits = sum(r["answer_ok"] for r in rows)
    p1 = hits / 100
    action = ("增加状态或位数" if p1 >= 0.99 else "进入小样本斜率验证" if 0.955 <= p1 <= 0.986
              else "弃用当前配置" if p1 < 0.85 else "偏低，微调配置后重新探针")
    return {"n": 100, "correct": hits, "p1": p1, "wilson95": wilson(hits, 100),
            "predicted_cot20_not_measured": p1**20,
            "in_probe_window": 0.955 <= p1 <= 0.986,
            "in_da1_descriptive_target": 0.80 <= p1 <= 0.95, "action": action,
            "parse_fail_rate": sum(not r["parse_ok"] for r in rows)/100,
            "trunc_rate": sum(r["trunc"] for r in rows)/100}

def slope_summary(rows):
    groups = defaultdict(list)
    for row in rows:
        if row["stage"] != "slope":
            raise ValueError("斜率摘要不能混入探针")
        groups[(row["question"]["d"], row["mode"])].append(row)
    expected_groups = {(d, m) for d in D_LIST for m in ("DA", "CoT")}
    if set(groups) != expected_groups or any(len(group) != 20 for group in groups.values()):
        raise ValueError("斜率阶段须完整覆盖 5 深度 × 20 题 × 2 模式")
    table, paired = [], {}
    for d in D_LIST:
        pair = {m: {r["question"]["qid"]: r for r in groups[d, m]} for m in ("DA", "CoT")}
        if len(pair["DA"]) != 20 or set(pair["DA"]) != set(pair["CoT"]):
            raise ValueError("DA / CoT 必须配对且无重复")
        paired[d] = [int(pair["CoT"][qid]["answer_ok"]) - int(pair["DA"][qid]["answer_ok"]) for qid in sorted(pair["DA"])]
        item = {"d": d}
        for mode in ("DA", "CoT"):
            group = groups[d, mode]
            hits = sum(r["answer_ok"] for r in group)
            item[mode] = {"n": 20, "correct": hits, "accuracy": hits/20, "wilson95": wilson(hits, 20),
                          "trunc_rate": sum(r["trunc"] for r in group)/20,
                          "parse_fail_rate": sum(not r["parse_ok"] for r in group)/20,
                          "mean_tokens": sum(r["n_tokens"] for r in group)/20}
        item["gap"] = sum(paired[d])/20
        table.append(item)
    trends = {m: ca_trend([r[m]["correct"] for r in table], [20]*5, D_LIST) for m in ("DA", "CoT")}
    rng = random.Random(SEED)
    boot = sorted(sum(rng.choices(paired[20], k=20))/20 - sum(rng.choices(paired[1], k=20))/20 for _ in range(5000))
    delta = table[-1]["gap"] - table[0]["gap"]
    checks = {"cot20_hard_target": 0.40 <= table[-1]["CoT"]["accuracy"] <= 0.75,
              "da20_target": table[-1]["DA"]["accuracy"] <= 0.10,
              "gap_growth_target": delta >= 0.30 - 1e-12,
              "cot_decreasing_ca_p_lt_001": trends["CoT"]["z"] < 0 and trends["CoT"]["p_two_sided"] < 0.01}
    evidence = [{"id": r["id"], "text": r["text"], "answer_ok": r["answer_ok"], "trunc": r["trunc"]} for r in sorted(rows, key=lambda r: r["id"])]
    return {"table": table, "ca_accuracy_trends": trends, "gap20_minus_gap1": delta,
            "gap_delta_paired_bootstrap95": [boot[124], boot[4874]],
            "gap_nondecreasing_observed": all(b["gap"] >= a["gap"] - 1e-12 for a, b in zip(table, table[1:])),
            "checks": checks, "evidence_digest": digest(evidence), "manual_review_required": True}

## 4. 离线自测（CPU 可运行，不写模型结果）

独立参考求解器逐步复算所有配置/深度/并行宽度；检查编码、边界、工作量和查表字段。
同时测试答案解析、统计退化情况、阶段完整性、人工审核门槛和断点续跑。
自测中的模拟模型只写临时目录，不能被正式定题流程接受。

In [7]:
def verify_question(question):
    if question.get("task_family") == "pointer_numeric":
        return verify_pointer_question(question)
    config = TaskConfig(**question["config"])
    actual_lengths = []
    for chain in question["chains"]:
        state = list(chain["initial"])
        assert all(0 <= v < config.base for v in state)
        for i, op in enumerate(chain["ops"]):
            # 独立参考实现：不调用 apply_op/encode_state，防止自测只重复被测实现。
            a, b = op["left"], op["right"]
            old_a, old_b = state[a], state[b]
            sign = 1 if old_a <= old_b else -1
            state[a], state[b] = old_a + sign*op["x"], old_b - sign*op["y"]
            assert op["x"] != op["y"] and a != b
            assert all(0 <= v < config.base for v in state)
            assert state == chain["trace"][i]
        assert state == chain["final"]
        assert sum(v*config.base**i for i, v in enumerate(reversed(state))) == chain["ans"]
        actual_lengths.append(len(chain["ops"]))
    assert question["d"] == max(actual_lengths)
    assert question["W"] == sum(actual_lengths)
    assert question["W_atomic"] == 3 * question["W"]
    assert question["w"] == len(actual_lengths)
    assert question["answers"] == [c["ans"] for c in question["chains"]]
    assert question["ans"] == ",".join(map(str, question["answers"]))
    assert question["lookup"]["OPS"] == actual_lengths
    for j, chain in enumerate(question["chains"]):
        assert question["lookup"]["FIRST"][j] == op_text(chain["ops"][0])
        assert question["lookup"]["LAST"][j] == op_text(chain["ops"][-1])
        assert question["lookup"]["START"][j] == sum(v*config.base**i for i, v in enumerate(reversed(chain["initial"])))
        assert question["lookup"]["MAXDELTA"][j] == max(v for op in chain["ops"] for v in (op["x"], op["y"]))
    if question["kind"] == "parallel":
        assert question["W"] == question["matched_serial_d"] == 2*question["w"]-1
    assert question["q"] == render_question(question["chains"], config)

def expect_error(error, fn):
    try:
        fn()
    except error:
        return
    raise AssertionError(f"预期 {error.__name__}")

def offline_core_tests():
    count = 0
    for config in CANDIDATES.values():
        if not isinstance(config, TaskConfig):
            continue
        for d in range(1, 21):
            for i in range(10):
                verify_question(make_question(config, d, i, "selftest"))
                count += 1
        for w in range(1, 11):
            for i in range(10):
                verify_question(make_question(config, 1 if w == 1 else 2, i, "selftest", "parallel", w))
                count += 1
    config = CANDIDATES["register2_small"]
    question = make_question(config, 1, 0, "selftest")
    assert question == make_question(config, 1, 0, "selftest")
    assert question["qid"] != make_question(config, 1, 0, "probe")["qid"]
    assert apply_op([5, 5], {"left": 0, "right": 1, "x": 2, "y": 3}) == [7, 2]
    assert apply_op([6, 5], {"left": 0, "right": 1, "x": 2, "y": 3}) == [4, 8]
    for text in (question["ans"], f"####ANSWER= {question['ans']}.0", f"trace\n####ANSWER={question['ans']}"):
        assert score_answer(text, question, "DA")["answer_ok"]
    for text in ("", "A=41 B=18", "####ANSWER=12 or 34", "####ANSWER=1.5", "####ANSWER=1,2"):
        assert parse_answer(text, 1) is None
    assert parse_answer("####ANSWER=1\n####ANSWER=2", 1) == [2]
    assert parse_answer("####ANSWER=1, 2", 2) == [1, 2]
    assert parse_answer("####ANSWER=1", 2) is None
    assert not score_answer("####ANSWER=-1", question, "DA")["answer_ok"]
    assert not score_answer(f"trace\n{question['ans']}", question, "DA")["base_ok"]
    assert ca_trend([20]*5, [20]*5, D_LIST)["p_two_sided"] == 1
    assert math.isclose(ca_trend([0, 10], [10, 10], [1, 2])["z"], math.sqrt(20))
    trend = ca_trend([20, 18, 14, 10, 8], [20]*5, D_LIST)
    assert trend["z"] < 0 and trend["p_two_sided"] < .01
    reverse = ca_trend([0, 2, 6, 10, 12], [20]*5, D_LIST)
    assert math.isclose(trend["z"], -reverse["z"])
    lo, hi = wilson(96, 100)
    assert .90 < lo < .96 < hi < 1
    expect_error(ValueError, lambda: probe_summary([]))
    expect_error(ValueError, lambda: slope_summary([]))
    expect_error(ValueError, lambda: make_question(config, 3, 0, "test", "parallel", 3))
    print(f"离线核心自测通过：{count} 道题逐步复算；无模型推理，无校准结论。")

offline_core_tests()

离线核心自测通过：1200 道题逐步复算；无模型推理，无校准结论。


## 本轮准备：恢复已完成的真实探针

已复核提交 `d2b57a4` 及 ZIP：delta 11/100，lookup 87/100，200 条均无输入 token 错配、解析失败或截断。
本轮直接验证 lookup 的深度曲线。其 0.87 符合 A.0 的 DA(1) 描述目标；探针窗口由 DA→CoT 的启发式估计而来，不能代替 CoT(20) 实测。
例外理由已写入 `PROBE_OVERRIDE_REASON`，不表示校准通过或允许生成正式数据。

从 GitHub 直接打开 Colab 不会带上 JSONL。本格在加载 GPU 前恢复 SHA256 固定的归档，逐题重建并复核判分后合并，
保留已有结果、拒绝冲突。已有完整匹配探针则跳过下载。网络失败时，可将原 ZIP 的 JSONL 放到 RESULT_PATH。
随后仍严格比较本轮模型/环境指纹；若环境不同，不将旧探针冒充同条件测量。

In [8]:
ARCHIVED_PROBE_SHA256 = "ef295e30b21cd10c5f89f77514274b8951fcc7e11e618adf99fea38eea069d0a"
ARCHIVED_PROBE_RELATIVE = "Experiments/task_a/archives/pointer_search_d2b57a4/calib_v3_task.jsonl"
ARCHIVED_PROBE_URL = "https://raw.githubusercontent.com/wangziwei2018/CISC7404-Project/taskA/" + ARCHIVED_PROBE_RELATIVE
ARCHIVED_PROBE_META = {'tier': 'B', 'model_id': 'Qwen/Qwen3-4B-Instruct-2507', 'model_commit': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'requested_model_revision': None, 'revision_kind': 'hub_commit', 'local_revision_label': None, 'loading': {'model': 'Qwen/Qwen3-4B-Instruct-2507', 'dtype': 'half', 'gpu_memory_utilization': 0.8, 'max_model_len': 8192, 'max_num_seqs': 1, 'enforce_eager': True, 'revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'tokenizer_revision': 'cdbee75f17c01a7cc42f958dc650907174af0554'}, 'chat_kwargs': {}, 'gpu': 'Tesla T4', 'top_k_passed': False, 'packages': {'vllm': '0.33.0', 'torch': '2.14.0', 'transformers': '5.20.2', 'datasets': '4.8.5', 'huggingface_hub': '1.33.0'}}


def restore_archived_pointer_probe(root=ROOT, result_path=RESULT_PATH, fetch=None):
    # 先检查本地已有断点。所有字段/判分一致时无需联网，不重跑 100 条已完成探针。
    config = CANDIDATES["pointer40_lookup"]
    expected = make_jobs([make_question(config, 1, i, "probe") for i in range(100)],
                         "probe", ("DA",), ARCHIVED_PROBE_META)
    result_path = Path(result_path)
    existing = index_results(result_path)
    if all(job["id"] in existing for job in expected):
        load_complete(expected, result_path)
        print("已有完整的 100 条 lookup 真实探针，校验通过；无需下载或重跑。")
        return 0
    archived = Path(root) / ARCHIVED_PROBE_RELATIVE
    if archived.is_file():
        payload = archived.read_bytes()
    else:
        if fetch is None:
            from urllib.request import urlopen
            def fetch(url):
                with urlopen(url, timeout=30) as response:
                    return response.read(2_000_001)
        try:
            payload = fetch(ARCHIVED_PROBE_URL)
        except Exception as exc:
            raise RuntimeError("无法恢复探针。请将已导出 ZIP 的 calib_v3_task.jsonl 放入 RESULT_PATH 后重跑；不要重新跑探针。") from exc
    if hashlib.sha256(payload).hexdigest() != ARCHIVED_PROBE_SHA256:
        raise ValueError("归档探针 SHA256 不匹配；拒绝导入，现有结果未改动。")
    # 在临时目录完整复核后才合并，禁止覆盖已有冲突记录或混入模拟结果。
    with tempfile.TemporaryDirectory() as tmp:
        source = Path(tmp) / "source.jsonl"
        source.write_bytes(payload)
        rows = read_jsonl(source)
        if len(rows) != 200 or len({row["id"] for row in rows}) != 200:
            raise ValueError("归档应为 200 条唯一真实指针探针")
        jobs = []
        for row in rows:
            q = row["question"]
            if row["run_meta"] != ARCHIVED_PROBE_META or row["stage"] != "probe" or row["mode"] != "DA":
                raise ValueError("归档来源或阶段不符")
            config = config_from_dict(q["config"])
            rebuilt = make_question(config, q["d"], q["i"], q["split"])
            if q != rebuilt:
                raise ValueError("归档题目无法重建")
            verify_question(q)
            jobs.extend(make_jobs([q], "probe", ("DA",), row["run_meta"]))
        load_complete(jobs, source)
        load_complete(expected, source)
        additions = []
        for row, line in zip(rows, payload.splitlines(keepends=True)):
            previous = existing.get(row["id"])
            if previous is not None and previous != row:
                raise ValueError("同一作业存在冲突结果，拒绝覆盖；请先人工复核。")
            if previous is None:
                additions.append(line)
        old = result_path.read_bytes() if result_path.exists() else b""
        if old and not old.endswith(b"\n"):
            old += b"\n"
        result_path.parent.mkdir(parents=True, exist_ok=True)
        target = result_path.with_suffix(".restore.tmp")
        try:
            target.write_bytes(old + b"".join(additions))
            load_complete(expected, target)
            target.replace(result_path)
        finally:
            target.unlink(missing_ok=True)
    print(f"已恢复 {len(additions)} 条真实探针；旧结果保留。本轮只新增 200 条斜率推理。")
    return len(additions)


if RUN_SLOPE and SELECTED == "pointer40_lookup" and RESTORE_ARCHIVED_PROBE:
    restore_archived_pointer_probe()
else:
    print("当前阶段无需自动恢复归档探针。")


已有完整的 100 条 lookup 真实探针，校验通过；无需下载或重跑。


## 5. 加载 vLLM（仅在真实推理阶段执行）

B 沿用用户本次已跑通的 T4 配置：half / 0.80 / 8192 / 1 / enforce_eager=True，
并设置 VLLM_ENABLE_V1_MULTIPROCESSING=0；不传 thinking kwargs。
该配置已完成探针与诊断；64 条诊断未发现输入错配，基础控题及最终答案的批量对照通过。
A 原样沿用 AGENTS 的 0.80 / 8192 模板，并显式 `enable_thinking=False`。
两种模式共用 6144 输出 token，生成原文不做后处理截断。每题独立采样种子使续跑不依赖批次顺序。
实际依赖版本、GPU、模型 commit、加载参数进入运行指纹，避免混用不同运行环境的结果。

远端模型在 **创建 LLM 前**，通过 Hugging Face 的 `model_info().sha` 把 main/tag/分支解析为
不可变 commit，并将同一 commit 传给 `revision` 和 `tokenizer_revision`。不读取
`hf_config._commit_hash` 等私有字段，默认 `MODEL_REVISION=None` 可直接运行。
完整 commit 可直接指定；本地 HF snapshot 从目录识别 commit，普通本地权重目录仍须提供
`MODEL_REVISION` 作为人工维护的权重版本标签（不会冒充已验证的远端 commit）。

若上一版已创建模型后才因版本记录报错，请先 **重新启动会话** 再运行本版，释放旧引擎的显存；
不要删除运行时，已安装依赖和下载缓存可以继续复用。

In [9]:
if RUN_SEARCH or RUN_DIAGNOSTIC or RUN_PROBES or RUN_SLOPE:
    import subprocess
    subprocess.run(["nvidia-smi"], check=True)

In [10]:
def configured_model_location():
    model_id = {"A": "Qwen/Qwen3-1.7B", "B": "Qwen/Qwen3-4B-Instruct-2507"}[TIER]
    local = ROOT / "models" / model_id.split("/")[-1]
    return model_id, local


def resolve_model_revision(model_id, local, requested_revision):
    if requested_revision is not None and (not isinstance(requested_revision, str) or not requested_revision.strip()):
        raise ValueError("MODEL_REVISION 须为 None 或非空的 commit/tag/版本标签")
    if local.is_dir():
        source = local.resolve()
        if source.parent.name == "snapshots" and re.fullmatch(r"[0-9a-fA-F]{40}", source.name):
            commit = source.name.lower()
            if requested_revision is not None and requested_revision.lower() != commit:
                raise ValueError("本地 snapshot 与 MODEL_REVISION 不匹配；请留空或填写该 snapshot 的完整 commit。")
            return {"source": str(source), "model_commit": commit,
                    "local_revision_label": None, "revision_kind": "local_snapshot", "loading_kwargs": {}}
        if requested_revision is None:
            raise RuntimeError("普通本地权重目录没有可验证的 HF commit；请在加载前填写 MODEL_REVISION 为权重版本标签。")
        return {"source": str(source), "model_commit": None,
                "local_revision_label": requested_revision, "revision_kind": "local_label", "loading_kwargs": {}}

    if requested_revision is not None and re.fullmatch(r"[0-9a-fA-F]{40}", requested_revision):
        commit = requested_revision.lower()
    else:
        from huggingface_hub import HfApi
        # 在 GPU 引擎创建前解析公开元数据，失败不会留下半初始化的 LLM。
        info = HfApi().model_info(repo_id=model_id, revision=requested_revision or "main")
        commit = info.sha
        if not isinstance(commit, str) or re.fullmatch(r"[0-9a-fA-F]{40}", commit) is None:
            raise RuntimeError("Hugging Face 未返回有效的模型 commit；请检查访问状态，或填写已知的完整 MODEL_REVISION。")
        commit = commit.lower()
    return {"source": model_id, "model_commit": commit,
            "local_revision_label": None, "revision_kind": "hub_commit",
            "loading_kwargs": {"revision": commit, "tokenizer_revision": commit}}


def load_vllm():
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # 必须先于 import vllm
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("本机没有 CUDA；保留离线验证，请在 Colab/T4 运行真实探针。")
    model_id, local = configured_model_location()
    if TIER == "B" and torch.cuda.get_device_properties(0).total_memory / 2**30 < 14:
        raise RuntimeError("档位 B 使用 T4/至少 14 GiB 可见显存；8GB 卡请用 A 做链路自测。")
    revision = resolve_model_revision(model_id, local, MODEL_REVISION)
    source = revision["source"]
    if TIER == "B":
        # 用户在 Colab/T4 已跑通的配置（c2cc018），继续沿用。
        loading = dict(model=source, dtype="half", gpu_memory_utilization=0.80,
                       max_model_len=8192, max_num_seqs=1, enforce_eager=True)
        chat_kw = {}
    else:
        loading = dict(model=source, gpu_memory_utilization=0.80, max_model_len=8192)
        chat_kw = {"chat_template_kwargs": {"enable_thinking": False}}
    loading.update(revision["loading_kwargs"])
    # 来源与软件版本均在分配 GPU 引擎前确认；不依赖 vLLM/transformers 的私有配置字段。
    meta = {"tier": TIER, "model_id": model_id, "model_commit": revision["model_commit"],
            "requested_model_revision": MODEL_REVISION, "revision_kind": revision["revision_kind"],
            "local_revision_label": revision["local_revision_label"],
            "loading": loading, "chat_kwargs": chat_kw,
            "gpu": torch.cuda.get_device_name(0), "top_k_passed": False,
            "packages": {p: importlib.metadata.version(p) for p in ("vllm", "torch", "transformers", "datasets", "huggingface_hub")}}
    print("加载前已确认模型来源：", json.dumps(meta, ensure_ascii=False, indent=2))
    from vllm import LLM, SamplingParams
    model = LLM(**loading)
    return model, SamplingParams, chat_kw, meta


if RUN_SEARCH or RUN_DIAGNOSTIC or RUN_PROBES or RUN_SLOPE:
    if "llm" not in globals():
        llm, SamplingParams, CHAT_KW, RUN_META = load_vllm()
    else:
        expected_id, expected_local = configured_model_location()
        expected_source = str(expected_local.resolve()) if expected_local.is_dir() else expected_id
        if (RUN_META["tier"] != TIER or RUN_META["model_id"] != expected_id
                or "requested_model_revision" not in RUN_META
                or RUN_META["requested_model_revision"] != MODEL_REVISION
                or RUN_META["loading"]["model"] != expected_source):
            raise RuntimeError("模型配置改变，请重启内核再加载，避免复用旧权重。")
    print(json.dumps(RUN_META, ensure_ascii=False, indent=2))
else:
    print("offline 模式：GPU 加载已跳过。真实推理请选择 RUN_STAGE='search'、'diagnostic'、'probe' 或 'slope' 后从头运行。")

加载前已确认模型来源： {
  "tier": "B",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "model_commit": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "requested_model_revision": null,
  "revision_kind": "hub_commit",
  "local_revision_label": null,
  "loading": {
    "model": "Qwen/Qwen3-4B-Instruct-2507",
    "dtype": "half",
    "gpu_memory_utilization": 0.8,
    "max_model_len": 8192,
    "max_num_seqs": 1,
    "enforce_eager": true,
    "revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554"
  },
  "chat_kwargs": {},
  "gpu": "Tesla T4",
  "top_k_passed": false,
  "packages": {
    "vllm": "0.33.0",
    "torch": "2.14.0",
    "transformers": "5.20.2",
    "datasets": "4.8.5",
    "huggingface_hub": "1.33.0"
  }
}


W1109 07:28:21.684000 15322 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1109 07:28:21.687000 15322 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1109 07:28:21.688000 15322 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=15322) INFO 11-09 07:28:39 [entrypoints/common/startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.8, 'max_num_seqs': 1, 'disable_log_stats': True, 'revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'tokenizer_revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
(MainProcess pid=15322) WARNING 11-09 07:28:39 [engine/arg_utils.py:1984] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=15322) DEBUG 11-09 07:28:40 [model_executor/models/registry.py:1109] Loaded model info for class vllm.model_executor.models.qwen3.Qwen3ForCausalLM from cache
(MainProcess pid=15322) DEBUG 11-09 07:28:40 [logging_utils/log_time.py:26] Registry inspect model class: Elapsed time 0.0028979 secs
(MainProcess pid=15322) INFO 11-09 07:28:40 [config/

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

(MainProcess pid=15322) WARNING 11-09 07:28:41 [config/vllm.py:1778] Enforce eager set, disabling torch.compile, CUDAGraphs, and JIT kernel warmup. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none and --kernel_config.enable_jit_warmup=False
(MainProcess pid=15322) WARNING 11-09 07:28:41 [config/vllm.py:1802] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(MainProcess pid=15322) DEBUG 11-09 07:28:41 [config/kernel.py:462] Setting platform-specific IR op priority defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'], gelu_and_mul_sparse=['triton', 'native']), user-defined: IrOpPriorityConfig(rms_norm=[], fused_add_rms_norm=[], gelu_and_mul_sparse=[])
(MainProcess pid=15322) INFO 11-09 07:28:41 [config/kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

(MainProcess pid=15322) DEBUG 11-09 07:28:42 [tokenizers/registry.py:87] Loading CachedHfTokenizer for tokenizer_mode='hf'
(MainProcess pid=15322) DEBUG 11-09 07:28:43 [renderers/registry.py:61] Loading HfRenderer for renderer_mode='hf'
(MainProcess pid=15322) DEBUG 11-09 07:28:45 [v1/worker/.../logits_processor/loader.py:48] No logitsprocs plugins installed (group vllm.logits_processors).
(MainProcess pid=15322) INFO 11-09 07:28:45 [v1/engine/core.py:139] Initializing a V1 LLM engine (v0.33.0) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=cdbee75f17c01a7cc42f958dc650907174af0554, tokenizer_revision=cdbee75f17c01a7cc42f958dc650907174af0554, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(MainProcess pid=15322) WARNING 11-09 07:28:57 [model_executor/models/utils.py:333] Attempted to load weight embed_tokens.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(MainProcess pid=15322) DEBUG 11-09 07:29:00 [model_executor/models/utils.py:343] Loaded weight embed_tokens.weight with shape torch.Size([151936, 2560])
(MainProcess pid=15322) WARNING 11-09 07:29:00 [model_executor/models/utils.py:333] Attempted to load weight layers.0.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(MainProcess pid=15322) DEBUG 11-09 07:29:00 [model_executor/models/utils.py:343] Loaded weight layers.0.input_layernorm.weight with shape torch.Size([2560])
(MainProcess pid=15322) WARNING 11-09 07:29:00 [model_executor/models/utils.py:333] Attempted to load weight layers.0.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(MainProcess pid=15322) DEBUG 11-09 07:29:00 [model_executor/models/utils.py:343]

In [11]:
def run_jobs(jobs, model, sampling_class, chat_kw, path=RESULT_PATH, backend="vllm", batch_size=None):
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    if not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("batch_size 必须为正整数")
    if any(j.get("execution", {}).get("batch_size", batch_size) != batch_size for j in jobs):
        raise ValueError("实际批次大小与作业记录不一致")
    path = Path(path)
    indexed = index_results(path)
    if len({job["id"] for job in jobs}) != len(jobs):
        raise ValueError("不能运行重复作业")
    for job in jobs:
        if job["id"] in indexed:
            previous = indexed[job["id"]]
            if previous.get("backend") != backend or any(previous.get(k) != v for k, v in job.items()):
                raise ValueError("已有结果与当前作业不一致，拒绝覆盖")
    pending = [job for job in jobs if job["id"] not in indexed]
    if not pending:
        print(f"已完成 {len(jobs)} 条，无需重复推理。")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer = model.get_tokenizer()
    for offset in range(0, len(pending), batch_size):
        batch = pending[offset:offset+batch_size]
        expected_tokens = []
        for job in batch:
            tokens = tokenizer.apply_chat_template(job["messages"], tokenize=True, add_generation_prompt=True,
                                                   **chat_kw.get("chat_template_kwargs", {}))
            # 兼容 tokenizer 返回 list、tensor 或 BatchEncoding。
            if hasattr(tokens, "keys"):
                tokens = tokens["input_ids"]
            if hasattr(tokens, "tolist"):
                tokens = tokens.tolist()
            if tokens and isinstance(tokens[0], list):
                if len(tokens) != 1:
                    raise ValueError("单条题目意外生成了多个 token 序列")
                tokens = tokens[0]
            tokens = list(tokens)
            expected_tokens.append(tokens)
            if len(tokens) + job["sampling"]["max_tokens"] > 8192:
                raise ValueError("题干与输出预算超过上下文；禁止自动截断，先修改设计并重校准。")
        params = [sampling_class(temperature=j["sampling"]["temperature"], top_p=j["sampling"]["top_p"],
                                 max_tokens=j["sampling"]["max_tokens"], seed=j["sampling_seed"]) for j in batch]
        outputs = model.chat([job["messages"] for job in batch], params, use_tqdm=False, **chat_kw)
        if len(outputs) != len(batch) or any(len(o.outputs) != 1 for o in outputs):
            raise RuntimeError("推理返回数量不匹配，当前批次未写入")
        with path.open("a", encoding="utf-8") as f:
            for job, output, expected in zip(batch, outputs, expected_tokens):
                result = output.outputs[0]
                actual_tokens = list(output.prompt_token_ids)
                input_check = {"request_id": str(getattr(output, "request_id", "unavailable")),
                               "input_tokens_match": actual_tokens == expected,
                               "expected_prompt_tokens": len(expected),
                               "actual_prompt_sha256": digest(actual_tokens), "expected_prompt_sha256": digest(expected)}
                if job["stage"] == "diagnostic":
                    input_check.update({"actual_prompt_token_ids": actual_tokens, "expected_prompt_token_ids": expected,
                                        "rendered_prompt": tokenizer.decode(actual_tokens, skip_special_tokens=False)})
                row = {**job, **input_check, "backend": backend, "text": result.text,
                       "finish_reason": result.finish_reason, "stop_reason": result.stop_reason,
                       "n_tokens": len(result.token_ids), "prompt_tokens": len(output.prompt_token_ids),
                       "trunc": result.finish_reason == "length", "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                       **score_answer(result.text, job["question"], job["mode"])}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
            os.fsync(f.fileno())
        print(f"新增 {min(offset+batch_size, len(pending))}/{len(pending)} 条")

def prepare_jobs(config, stage, run_meta):
    if stage == "probe":
        questions = [make_question(config, 1, i, "probe") for i in range(100)]
        modes = ("DA",)
    elif stage == "slope":
        questions = [make_question(config, d, i, "slope") for d in D_LIST for i in range(20)]
        modes = ("DA", "CoT")
    else:
        raise ValueError("未知阶段")
    path = WORK_DIR / f"{stage}_{digest([VERSION, SEED, asdict(config)])[:16]}.jsonl"
    loaded = roundtrip_questions(path, questions)
    assert loaded == questions
    return make_jobs(loaded, stage, modes, run_meta)

## 6 前置：低准确率诊断（已完成，64 条；可选复跑）

标准配置 0/100、简化配置 1/100：逐条复算后未发现生成器/gold/判分错误。
简化配置有 18/100 个答案超出两位编码范围，15/100 直接等于初态编码。
这些现象尚不能区分模型能力与推理链路，不能直接声称任务族或硬件有问题。

16 个固定案例包含抄写、读寄存器、加减、比较、独立编码、条件更新后返回两个值，以及原题编码。
每个案例在 DA/CoT 下分别单条提交与批量提交（1 / 16），共 64 条；对照使用相同题目和每题采样 seed。
T4 引擎仍是 max_num_seqs=1，因此比较的是 API 提交批量/调度路径，不是 16 个序列同时在 GPU 计算。
任务采样参数不变。比较原始文本是否一致、正确率、解析/截断以及**预期模板 token 与实际输入 token**，
保存实际 token、解码后的题干和 request_id，避免仅凭题目旁的回答猜测原因。

本小样本用于排查，不估计正式 p1，也不代替后续 100 题探针或 200 条斜率验证。


In [12]:
def diagnostic_questions():
    cases = []
    def add(kind, label, prompt, answers, source_qid=None):
        key = {"version": VERSION, "split": "diagnostic_v1", "kind": kind, "label": label,
               "q": prompt, "answers": answers, "source_qid": source_qid}
        cases.append({**key, "qid": digest(key), "diagnostic_kind": kind, "w": 1,
                      "answer_width": len(answers), "ans": ",".join(map(str, answers))})
    for number in (17, 83):
        add("copy", str(number), f"Return this integer unchanged: {number}.", [number])
    for a, b in ((7, 3), (41, 18)):
        add("read", f"{a}-{b}", f"A={a}, B={b}. What is the value of B?", [b])
    add("arithmetic", "add", "Compute 7 + 2. Return the result.", [9])
    add("arithmetic", "subtract", "Compute 41 - 6. Return the result.", [35])
    add("compare", "greater", "If 41 > 18, return 7; otherwise return 3.", [7])
    add("compare", "equal", "If 5 > 5, return 7; otherwise return 3.", [3])
    for a, b, base in ((7, 3, 10), (41, 18, 100)):
        add("encoding", str(base), f"A={a}, B={b}. Compute A*{base} + B.", [a*base+b])
    for name in ("register2_easy", "register2_small"):
        for i in range(2):
            original = make_question(CANDIDATES[name], 1, i, "probe")
            add("register_encoded", f"{name}-{i}", original["q"], original["answers"], original["qid"])
            if name == "register2_easy":
                prompt = original["q"].split("Encode the final registers", 1)[0]
                prompt += "Return the final value of A followed by the final value of B, separated by a comma."
                add("register_pair", str(i), prompt, original["chains"][0]["final"], original["qid"])
    assert len(cases) == 16
    return cases


def diagnostic_jobs(questions, run_meta, batch_size):
    execution = {"batch_size": batch_size, "purpose": "diagnostic_v1",
                 "runtime_env_observed": {name: os.environ.get(name) for name in (
                     "VLLM_USE_FLASHINFER_SAMPLER", "VLLM_ENABLE_V1_MULTIPROCESSING", "VLLM_LOGGING_LEVEL")}}
    return make_jobs(questions, "diagnostic", ("DA", "CoT"), run_meta, execution=execution)


def diagnostic_summary(rows):
    expected = {(q["qid"], mode, size) for q in diagnostic_questions() for mode in ("DA", "CoT") for size in (1, 16)}
    actual = {(r["question"]["qid"], r["mode"], r["execution"]["batch_size"]) for r in rows}
    if len(rows) != 64 or actual != expected:
        raise ValueError("诊断须完整覆盖 16 案例 × 2 模式 × 2 提交批量")
    groups = defaultdict(list)
    paired = defaultdict(dict)
    for row in rows:
        size = row["execution"]["batch_size"]
        groups[(row["question"]["diagnostic_kind"], row["mode"], size)].append(row)
        paired[(row["question"]["qid"], row["mode"])][size] = row
    table = []
    for (kind, mode, size), group in sorted(groups.items()):
        table.append({"kind": kind, "mode": mode, "batch_size": size, "n": len(group),
                      "correct": sum(r["answer_ok"] for r in group),
                      "input_token_mismatches": sum(r.get("input_tokens_match") is not True for r in group),
                      "parse_failures": sum(not r["parse_ok"] for r in group),
                      "truncations": sum(r["trunc"] for r in group)})
    differences = [{"qid": qid, "mode": mode, "single_text": pair[1]["text"], "batch_text": pair[16]["text"]}
                   for (qid, mode), pair in paired.items() if pair[1]["text"] != pair[16]["text"]]
    core = [r for r in rows if r["question"]["diagnostic_kind"] in ("copy", "read", "arithmetic", "compare")]
    mismatches = sum(r.get("input_tokens_match") is not True for r in rows)
    core_errors = sum(not r["answer_ok"] for r in core)
    if mismatches:
        next_action = "存在输入 token 不一致，先核对实际题干与请求对应关系，不进行难度校准。"
    elif core_errors:
        next_action = "基础控题出现错误；先复核原文、运行链路和模式差异，不能仅归因为深度任务太难。"
    else:
        next_action = "基础控题通过；结合编码/条件更新及 DA/CoT 差异调整任务，或尝试编号数组指针，再重跑 100 题探针。"
    return {"version": VERSION, "stage": "diagnostic", "n": len(rows), "table": table,
            "input_token_mismatches": mismatches, "core_errors": core_errors,
            "submission_pair_text_differences": differences, "next_action": next_action,
            "ready_for_slope": False, "formal_generation_approved": False}


if RUN_DIAGNOSTIC:
    questions = roundtrip_questions(WORK_DIR / "diagnostic_v1_questions.jsonl", diagnostic_questions())
    for size in (1, 16):
        jobs = diagnostic_jobs(questions, RUN_META, size)
        CURRENT_JOBS = list({j["id"]: j for j in CURRENT_JOBS + jobs}.values())
        run_jobs(jobs, llm, SamplingParams, CHAT_KW, path=DIAGNOSTIC_PATH, batch_size=size)
    diagnostic_rows = load_complete(CURRENT_JOBS, DIAGNOSTIC_PATH)
    report = diagnostic_summary(diagnostic_rows)
    report["run_meta"] = RUN_META
    DIAGNOSTIC_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print("64/64 条诊断完成。它不是正式 p1；请导出结果供复核后再决定下一步。")
    for row in [r for r in diagnostic_rows if not r["answer_ok"]][:4]:
        print({"kind": row["question"]["diagnostic_kind"], "mode": row["mode"],
               "batch_size": row["execution"]["batch_size"], "gold": row["question"]["ans"],
               "text_preview": row["text"][:1500], "preview_only": True})
else:
    print("当前未运行诊断阶段。")

当前未运行诊断阶段。


## 自动校准：编号数组指针（上一轮策略，保留复现）

规则在运行前固定，最多 3 个配置各 100 条 DA 探针；**不先跑深度矩阵**。

- 起点 `pointer40_delta`（编号 0..39）。p1≥0.99 时试三位编号 `pointer40_delta3`；p1<0.955 时试只查表的 `pointer40_lookup`。
- 三位 delta 仍过易时打乱表行；若偏难则试全两位编号 10..49。
- 只查表过易时打乱表行；仍偏难则停止，不自动选择失败配置。
- 任一配置 p1∈[0.955,0.986]，立即用新的 `slope` split 跑 d=1,5,10,15,20 ×20×DA/CoT，最多 200 条，然后停止供审核。

因此一次运行最多 300 条探针 + 200 条斜率；未命中窗口时只输出失败记录和下一步建议。
100 题的 p1 有较大置信区间；自动选择只是推进小样本，不是确认任务有效，更不会批准正式生成。
若斜率未达标，不继续自动搜索下一任务以避免无上限试错。断点以原始 JSONL 为准，重跑会跳过已完成作业。


In [13]:
def next_pointer_candidate(name, p1):
    if 0.955 <= p1 <= 0.986:
        return None
    harder = p1 >= 0.99
    if name == "pointer40_delta":
        return "pointer40_delta3" if harder else "pointer40_lookup"
    if name == "pointer40_delta3":
        return "pointer40_delta3_shuffled" if harder else "pointer40_delta2"
    if name == "pointer40_lookup" and harder:
        return "pointer40_lookup_shuffled"
    return None


def search_pointer_candidates(run_meta, run, prepare, load, save, max_probes=SEARCH_MAX_PROBES, start=SEARCH_START):
    if not isinstance(max_probes, int) or not 1 <= max_probes <= 3 or start not in POINTER_CANDIDATES:
        raise ValueError("search 须从已声明的指针配置开始，最多测试 1..3 个配置")
    report = {"version": POINTER_VERSION, "stage": "search", "run_meta": run_meta, "start": start,
              "max_probe_configs": max_probes, "trials": [], "selected": None,
              "status": "probing", "formal_generation_approved": False}
    name, seen = start, set()
    for _ in range(max_probes):
        if name is None or name in seen:
            break
        seen.add(name)
        config = POINTER_CANDIDATES[name]
        jobs = prepare(config, "probe", run_meta)
        run(jobs)
        summary = probe_summary(load(jobs))
        report["trials"].append({"candidate": name, "config": asdict(config), "probe": summary})
        save(report)
        print(name, json.dumps(summary, ensure_ascii=False, indent=2))
        if summary["in_probe_window"]:
            report.update(selected=name, status="slope_pending")
            save(report)
            jobs = prepare(config, "slope", run_meta)
            run(jobs)
            slope = slope_summary(load(jobs))
            report.update(slope=slope, status="awaiting_human_review" if slope["checks"]["cot20_hard_target"] else "slope_target_missed")
            save(report)
            return report
        name = next_pointer_candidate(name, summary["p1"])
    report.update(status="no_candidate_in_window", next_candidate=name if name not in seen else None)
    save(report)
    return report


if RUN_SEARCH:
    def run_search_jobs(jobs):
        global CURRENT_JOBS
        CURRENT_JOBS = list({j["id"]: j for j in CURRENT_JOBS + jobs}.values())
        run_jobs(jobs, llm, SamplingParams, CHAT_KW)

    def save_search_report(report):
        SEARCH_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
        SEARCH_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    search_report = search_pointer_candidates(RUN_META, run_search_jobs, prepare_jobs, load_complete, save_search_report)
    print(json.dumps(search_report, ensure_ascii=False, indent=2))
    if search_report["selected"] is not None:
        SELECTED = search_report["selected"]
        selected_slope = search_report["slope"]
        selected_probe = next(t["probe"] for t in search_report["trials"] if t["candidate"] == SELECTED)
        report = {"version": POINTER_VERSION, "candidate": SELECTED, "config": asdict(CANDIDATES[SELECTED]),
                  "run_meta": RUN_META, "probe": selected_probe, "slope": selected_slope,
                  "selection": "bounded_pointer_search"}
        SLOPE_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        print("小样本已完成，请人工审查原文和 CoT(20)/gap/趋势；正式生成仍关闭。")
    else:
        print("本轮没有配置进入 p1 窗口；已保留所有失败记录，不跑斜率、不生成正式数据。")
else:
    print("当前未运行自动指针校准。")

当前未运行自动指针校准。


## 6. A.1 探针：先首选，再调难度

手动 probe 模式默认 `pointer40_delta`；search 模式按上节规则运行；本轮默认 slope，复用已归档探针。
寄存器两配置已明确失败，保留代码与历史数据供复核，不再默认重跑它们。
指针 delta=11/100、lookup=87/100 已实测；后者的 CoT 深度曲线尚待验证。

In [14]:
if RUN_PROBES:
    for name in PROBE_CANDIDATES:
        jobs = prepare_jobs(CANDIDATES[name], "probe", RUN_META)
        CURRENT_JOBS = list({j["id"]: j for j in CURRENT_JOBS + jobs}.values())
        run_jobs(jobs, llm, SamplingParams, CHAT_KW)
        if TIER == "B":
            summary = probe_summary(load_complete(jobs))
            PROBE_REPORTS[name] = {"config": asdict(CANDIDATES[name]), "summary": summary}
            report = {"version": VERSION, "stage": "probe", "run_meta": RUN_META,
                      "candidates": PROBE_REPORTS, "formal_generation_approved": False}
            PROBE_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
            PROBE_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
            print(name, json.dumps(summary, ensure_ascii=False, indent=2))
            print(f"真实探针完成：{len(jobs)}/100 条；p1={summary['p1']:.3f}。下一步：{summary['action']}。")
        else:
            print("档位 A 仅链路自测；不能使用这些结果定题。")
else:
    print(f"当前阶段 {RUN_STAGE}：未执行新的探针。")

当前阶段 slope：未执行新的探针。


## 7. A.3 小样本斜率验证

必须先完成所选配置的 100 条 B 档探针。窗口外若仍值得探索，在 `PROBE_OVERRIDE_REASON`
记录理由；这只是允许跑 200 条小样本，不代表批准正式生成。
本轮默认 slope，lookup 探针会在加载 GPU 前自动恢复；例外理由已填写并保存进斜率报告。
不重跑探针，新增 200 条结果。仅保存 notebook 不会保存这些新结果，请下载最后一格 ZIP。

In [15]:
if RUN_SLOPE:
    probe_jobs = prepare_jobs(CANDIDATES[SELECTED], "probe", RUN_META)
    selected_probe = probe_summary(load_complete(probe_jobs))
    if not selected_probe["in_probe_window"] and not PROBE_OVERRIDE_REASON.strip():
        raise RuntimeError("p1 不在 0.955–0.986；先微调，或填写继续小样本的人工理由。")
    slope_jobs = prepare_jobs(CANDIDATES[SELECTED], "slope", RUN_META)
    CURRENT_JOBS = list({j["id"]: j for j in CURRENT_JOBS + slope_jobs}.values())
    run_jobs(slope_jobs, llm, SamplingParams, CHAT_KW)
    slope_rows = load_complete(slope_jobs)
    selected_slope = slope_summary(slope_rows)
    report = {"version": POINTER_VERSION if isinstance(CANDIDATES[SELECTED], PointerConfig) else VERSION, "candidate": SELECTED, "config": asdict(CANDIDATES[SELECTED]),
              "run_meta": RUN_META, "probe": selected_probe, "slope": selected_slope,
              "probe_override_reason": PROBE_OVERRIDE_REASON}
    SLOPE_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    SLOPE_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print("人工审核：检查摘要以及下方样例；完整原文见 calib_v3_task.jsonl。")
    for d in D_LIST:
        for mode in ("DA", "CoT"):
            group = [r for r in slope_rows if r["question"]["d"] == d and r["mode"] == mode]
            for target in (True, False):
                row = next((r for r in group if r["answer_ok"] == target), None)
                if row:
                    print(f"\nd={d} {mode} answer_ok={target} gold={row['question']['ans']} qid={row['question']['qid']}")
                    print(row["text"])
else:
    print(f"当前阶段 {RUN_STAGE}：未执行新的斜率验证；须完成探针后才进入 slope 阶段。")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

新增 8/8 条
{
  "version": "pointer_numeric_v1",
  "candidate": "pointer40_lookup",
  "config": {
    "name": "pointer40_lookup",
    "table_size": 40,
    "index_offset": 0,
    "operation": "lookup",
    "shuffle_rows": false,
    "family": "pointer_numeric"
  },
  "run_meta": {
    "tier": "B",
    "model_id": "Qwen/Qwen3-4B-Instruct-2507",
    "model_commit": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "requested_model_revision": null,
    "revision_kind": "hub_commit",
    "local_revision_label": null,
    "loading": {
      "model": "Qwen/Qwen3-4B-Instruct-2507",
      "dtype": "half",
      "gpu_memory_utilization": 0.8,
      "max_model_len": 8192,
      "max_num_seqs": 1,
      "enforce_eager": true,
      "revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
      "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554"
    },
    "chat_kwargs": {},
    "gpu": "Tesla T4",
    "top_k_passed": false,
    "packages": {
      "vllm": "0.33.0",
      "torch": "2.14.0",


## 8. 人工审核与正式生成（默认锁定）

**提醒开发者：task.md A.0 明确要求出结果后人工审查。** 检查 100 条探针、配对斜率曲线、
错误样例、截断/解析失败率、是否存在捷径，以及并行多答案读出成本。
CoT(20)∈[0.40,0.75] 保留为硬门槛；其余目标未满足时必须在 note 解释选择理由。
批准绑定到原始结果摘要指纹，换模型/配置/结果后需重新审核。

审查后在此 cell 下方自行填写（不要照抄空值，`approved` 默认为 False）：

```python
REVIEW = {
    "approved": False,
    "reviewer": "",
    "candidate": SELECTED,
    "evidence_digest": selected_slope["evidence_digest"],
    "note": "",  # 写清错误模式、未达目标及并行对照混杂的审查结论
}
GENERATE_FINAL = True
```

若重启内核后仅生成数据，不需要重新加载 GPU：从 `Experiments/task_a/task_a_summary.json` 读取
`run_meta` 赋给 `RUN_META`，重建对应作业并校验已有原文即可。正式 JSONL 的 `ans` 一律为字符串，
`answers` 为整数列表；`d` 永远表示实际最长依赖链，配平用 `matched_serial_d` / `W`。
生成后用 datasets 回读，再独立复算；汇总必须回填 record.md 与 AGENTS.md 5.5。
指针任务正式文件名为 `synthesis_pointer_numeric_dataset.jsonl`。若重启后只做生成，
从真实 search/斜率报告恢复 SELECTED 和 RUN_META，选择 offline，保留 REVIEW 的结果指纹；不必重新推理。

In [16]:
def require_review(review, candidate, summary):
    if not summary["checks"]["cot20_hard_target"]:
        raise ValueError("CoT(d=20) 不在硬目标区间；继续调题，不能正式生成。")
    if not isinstance(review, dict) or review.get("approved") is not True:
        raise ValueError("缺少人工批准；不得先造正式数据集。")
    if review.get("candidate") != candidate or review.get("evidence_digest") != summary["evidence_digest"]:
        raise ValueError("人工审核与当前候选/结果不匹配")
    if not str(review.get("reviewer") or "").strip() or not str(review.get("note") or "").strip():
        raise ValueError("请填写审查人及审查结论")

def formal_questions(config, n):
    if not isinstance(n, int) or n < 1:
        raise ValueError("每格题数必须为正整数")
    rows = [make_question(config, d, i, "formal") for d in range(1, 21) for i in range(n)]
    rows.extend(make_question(config, 1 if w == 1 else 2, i, "formal", "parallel", w)
                for w in range(1, 11) for i in range(n))
    return rows

def generate_final(config, run_meta, review, n=N_FINAL, result_path=RESULT_PATH, output_path=None):
    probe_jobs = prepare_jobs(config, "probe", run_meta)
    slope_jobs = prepare_jobs(config, "slope", run_meta)
    probe = probe_summary(load_complete(probe_jobs, result_path))
    slope = slope_summary(load_complete(slope_jobs, result_path))
    require_review(review, config.name, slope)
    rows = formal_questions(config, n)
    calibration_qids = {j["question"]["qid"] for j in probe_jobs+slope_jobs}
    calibration_prompts = {j["question"]["q"] for j in probe_jobs+slope_jobs}
    if any(r["qid"] in calibration_qids or r["q"] in calibration_prompts for r in rows):
        raise ValueError("正式题与校准题重叠，请调整正式 split 后再生成")
    if len({r["q"] for r in rows}) != len(rows):
        raise ValueError("正式题重复；请调整正式 split 或题量后再生成")
    for row in rows:
        verify_question(row)
    # 防止无意覆盖其它配置的正式产物。相同题目重复生成可安全复用。
    filename = "synthesis_pointer_numeric_dataset.jsonl" if isinstance(config, PointerConfig) else "synthesis_register_dataset.jsonl"
    path = Path(output_path) if output_path is not None else ROOT / "Experiments" / "data_generation" / filename
    if path.exists() and read_jsonl(path) != rows:
        raise FileExistsError("正式文件已存在且内容不同，请先归档旧版或使用新输出路径")
    loaded = roundtrip_questions(path, rows)
    assert loaded == rows
    for row in loaded:
        verify_question(row)
    manifest = {"version": POINTER_VERSION if isinstance(config, PointerConfig) else VERSION, "config": asdict(config), "run_meta": run_meta,
                "review": review, "probe": probe, "slope": slope, "n_per_cell": n,
                "serial_rows": 20*n, "parallel_rows": 10*n, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    path.with_suffix(".manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"正式生成并回读验证 {len(loaded)} 条：串行 {20*n}，并行 {10*n}。")
    return manifest

if GENERATE_FINAL:
    if "RUN_META" not in globals():
        raise RuntimeError("请从真实运行摘要恢复 RUN_META，不能凭空构造实验来源。")
    final_manifest = generate_final(CANDIDATES[SELECTED], RUN_META, REVIEW)
else:
    print("正式生成关闭；无实测与人工审核时不创建寄存器或指针正式数据集。")

正式生成关闭；无实测与人工审核时不创建寄存器或指针正式数据集。


In [17]:
def offline_pipeline_tests():
    from types import SimpleNamespace
    config = CANDIDATES["register2_small"]
    question = make_question(config, 1, 0, "pipeline_test")
    meta = {"tier": "B", "test_only": True}
    jobs = make_jobs([question], "probe", ("DA", "CoT"), meta)
    assert len(jobs) == 2 and jobs[0]["question"] == jobs[1]["question"]
    assert jobs[0]["messages"][1] == {"role": "user", "content": question["q"]}
    assert jobs[0]["id"] != make_jobs([question], "probe", ("DA",), {"tier": "A"})[0]["id"]

    class FakeModel:
        calls = 0
        def get_tokenizer(self):
            return SimpleNamespace(apply_chat_template=lambda *args, **kwargs: [1, 2, 3])
        def chat(self, messages, params, **kwargs):
            self.calls += 1
            assert all("top_k" not in p for p in params)
            return [SimpleNamespace(prompt_token_ids=[1, 2, 3], outputs=[SimpleNamespace(
                text=f"####ANSWER={question['ans']}", finish_reason="stop", stop_reason=None, token_ids=[4, 5])]) for _ in messages]

    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "mock.jsonl"
        model = FakeModel()
        run_jobs(jobs[:1], model, dict, {}, path, backend="offline_mock")
        run_jobs(jobs, model, dict, {}, path, backend="offline_mock")
        run_jobs(jobs, model, dict, {}, path, backend="offline_mock")
        assert model.calls == 2 and len(read_jsonl(path)) == 2
        expect_error(ValueError, lambda: load_complete(jobs, path))
        expect_error(RuntimeError, lambda: load_complete(jobs, Path(tmp)/"missing.jsonl"))
        expect_error(ValueError, lambda: run_jobs(jobs+jobs, model, dict, {}, path, backend="offline_mock"))
        class LongPromptModel(FakeModel):
            def get_tokenizer(self):
                return SimpleNamespace(apply_chat_template=lambda *args, **kwargs: [1]*2049)
        expect_error(ValueError, lambda: run_jobs(jobs, LongPromptModel(), dict, {}, Path(tmp)/"long.jsonl", backend="offline_mock"))
        duplicate_path = Path(tmp)/"duplicate.jsonl"
        write_jsonl(duplicate_path, read_jsonl(path)*2)
        expect_error(ValueError, lambda: index_results(duplicate_path))
        # JSONL → datasets 的嵌套字段、答案类型和工作量回读验证。
        sample = formal_questions(config, 2)
        assert len(sample) == 60
        loaded = roundtrip_questions(Path(tmp)/"questions.jsonl", sample)
        assert loaded == sample
        for row in loaded:
            verify_question(row)
        path.write_text('{"id":', encoding="utf-8")
        expect_error(ValueError, lambda: read_jsonl(path))

    summary = {"checks": {"cot20_hard_target": True}, "evidence_digest": "test-only"}
    review = {"approved": True, "reviewer": "离线测试", "note": "不是真实审核", "candidate": config.name, "evidence_digest": "test-only"}
    expect_error(ValueError, lambda: require_review(None, config.name, summary))
    expect_error(ValueError, lambda: require_review({**review, "evidence_digest": "old"}, config.name, summary))
    expect_error(ValueError, lambda: require_review(review, config.name, {**summary, "checks": {"cot20_hard_target": False}}))
    require_review(review, config.name, summary)
    rows = []
    for depth, da_hits, cot_hits in zip(D_LIST, (18, 10, 4, 1, 0), (20, 18, 16, 13, 10)):
        for i in range(20):
            q = make_question(config, depth, i, "stats_test")
            for mode, hits in (("DA", da_hits), ("CoT", cot_hits)):
                rows.append({"id": f"{depth}-{i}-{mode}", "stage": "slope", "mode": mode, "question": q,
                             "text": "模拟数据，绝不落盘", "answer_ok": i < hits, "trunc": False, "parse_ok": True, "n_tokens": 1})
    result = slope_summary(rows)
    assert all(result["checks"].values()) and math.isclose(result["gap20_minus_gap1"], .4)
    expect_error(ValueError, lambda: slope_summary(rows[:-1]))
    print("离线流程自测通过：断点续跑、去重、回读、配对统计、拒绝模拟结果与审核门槛。")

offline_pipeline_tests()

新增 1/1 条
新增 1/1 条
已完成 2 条，无需重复推理。


Generating train split: 0 examples [00:00, ? examples/s]

离线流程自测通过：断点续跑、去重、回读、配对统计、拒绝模拟结果与审核门槛。


## 9. 运行后的交付清单

- **不要只上传 notebook**。下载下方结果包，解压后把 JSONL、摘要和导出清单提交到 GitHub。
- 将真实 `calib_v3_task.jsonl` 连同 `Experiments/task_a/task_a_summary.json` 保存；保留失败配置，不能只交入选题。
- **人工审查** p1 / 曲线 / 错误原文及并行定义；没有通过就继续探针，不生成正式文件。
- 通过后保存当前任务族的正式 JSONL（指针为 `synthesis_pointer_numeric_dataset.jsonl`）与同名 `.manifest.json`；默认 2000 串行 + 1000 并行。
- 将模型 commit、参数、样本数、p1 CI、CoT(20)、DA(20)、gap 差、CA 检验对象和 p 值回填
  `record.md` 与 `AGENTS.md` 5.5；把 task.md 对应复选框标为完成。
- 指针字段为 START/OPS/TABLE_SIZE/INDEX_MIN/INDEX_MAX/FIRST/LAST；MAXDELTA 仅用于历史寄存器。Task B 须重定义并重标定。
- 不运行完整 k 矩阵；本次代码交付不是校准通过结论。

## 10. 真实结果核对与导出

只保存 notebook 不能交付模型原文。本格核对当前阶段每条作业与真实 JSONL 的一致性（包括诊断的输入核对记录），
输出完成数量和下一步，并生成 `task_a_results.zip`；Colab 默认触发下载。
ZIP 中保留仓库相对路径，将其解压后的 `Experiments/task_a/` 内容提交到 `taskA`。
如果中途推理失败，可以单独运行本格保存已完成的部分；未完成状态会明确标记，不能当作完整校准。
`offline` 模式不会生成结果包，模拟自测不会增加真实记录数。

In [18]:
def export_result_bundle(jobs, stage, root=ROOT, result_path=RESULT_PATH, report_paths=()):
    import zipfile

    if stage not in ("search", "diagnostic", "probe", "slope") or not jobs:
        raise ValueError("没有当前阶段的真实推理作业，不生成结果包。")
    allowed_stages = {"probe", "slope"} if stage == "search" else {stage}
    if any(job["stage"] not in allowed_stages for job in jobs):
        raise ValueError("导出作业阶段与 RUN_STAGE 不一致，请重新运行配置 cell。")
    indexed = index_results(result_path)
    completed_jobs = [job for job in jobs if job["id"] in indexed]
    if not completed_jobs:
        raise RuntimeError("当前阶段尚无真实推理结果；离线自测不算探针。")
    # 复算判分、核对提示词/采样/模型版本，拒绝混入模拟结果。
    load_complete(completed_jobs, result_path)
    if any(row.get("backend") != "vllm" or row["run_meta"].get("tier") != "B" for row in indexed.values()):
        raise ValueError("结果文件混有非 B 档真实推理记录，请分开归档后再导出。")
    if len({job["id"] for job in jobs}) != len(jobs):
        raise ValueError("当前导出作业有重复 id。")
    root = Path(root).resolve()
    result_path = Path(result_path).resolve()
    result_path.relative_to(root)
    archive_path = result_path.parent / "task_a_results.zip"
    manifest_path = result_path.parent / "task_a_export_manifest.json"
    manifest = {"version": VERSION, "question_versions": sorted({j["question"].get("version", VERSION) for j in jobs}),
                "stage": stage, "expected_current_jobs": len(jobs),
                "completed_current_jobs": len(completed_jobs), "stage_complete": len(completed_jobs) == len(jobs),
                "total_saved_real_results": len(indexed), "current_job_ids": [job["id"] for job in jobs],
                "result_sha256": hashlib.sha256(result_path.read_bytes()).hexdigest(),
                "formal_generation_approved": False}
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    payload = [result_path, manifest_path]
    payload.extend(Path(p).resolve() for p in report_paths if Path(p).is_file())
    for item in payload:
        item.relative_to(root)
    tmp = archive_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in payload:
            archive.write(item, arcname=item.relative_to(root).as_posix())
    tmp.replace(archive_path)
    print(f"真实 {stage} 结果：{len(completed_jobs)}/{len(jobs)} 条；完整阶段：{manifest['stage_complete']}。")
    print(f"结果包：{archive_path}。请解压并将 Experiments/task_a/ 下的结果文件提交到 GitHub。")
    return archive_path, manifest

if RUN_STAGE == "offline":
    print("本次没有新的模型推理；离线自测本身不能判断 Task A 是否达标。")
elif TIER != "B":
    print("档位 A 仅链路自测，不导出为 B 档正式校准结果。")
elif CURRENT_JOBS:
    export_path = DIAGNOSTIC_PATH if RUN_DIAGNOSTIC else RESULT_PATH
    extra_files = [PROBE_REPORT_PATH, SLOPE_REPORT_PATH, DIAGNOSTIC_REPORT_PATH, SEARCH_REPORT_PATH]
    if RUN_DIAGNOSTIC:
        extra_files.append(RESULT_PATH)  # 若本地保留历史探针，一并打包；不覆盖原文。
    bundle_path, export_manifest = export_result_bundle(
        CURRENT_JOBS, RUN_STAGE, result_path=export_path, report_paths=extra_files,
    )
    if AUTO_DOWNLOAD and Path("/content").is_dir():
        try:
            from google.colab import files
        except ImportError:
            print("非 Colab 环境，请从上述路径取走 ZIP。")
        else:
            files.download(str(bundle_path))
else:
    print("当前阶段尚未提交任何推理作业；请先运行对应的探针/斜率 cell。")

真实 slope 结果：200/200 条；完整阶段：True。
结果包：/content/taskA/Experiments/task_a/task_a_results.zip。请解压并将 Experiments/task_a/ 下的结果文件提交到 GitHub。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>